In [18]:
import os

import pandas as pd

from analysis.analysis_utils import generate_raw_csvs, generate_combined_df, post_process_df

pd.options.plotting.backend = "plotly"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
# Define WandB Parameters
entity = "phys_inversion"
project_prefix = "LASTMIN_last_minute"
filters = {"state": {"$eq": "finished"}}

rename_metrics = {
    "env/Env reach_success": "reach_success",
    "env/Env reach_distance": "reach_distance",
    "core/Episode Reward": "episode_reward"
}

order = ['MaskedMimic_Inversion_prior_False',
         'MaskedMimic_Inversion_prior_True',
         'MaskedMimic_Prior_Only_prior_True',
         'MaskedMimic_Finetune_prior_False',
         'MaskedMimic_Finetune_prior_True',
         'PULSE_prior_False',
         'AMP_prior_False',
         'PureRL_prior_False',
         'PPO_prior_False', ]

relevant_algos = ['MaskedMimic_Inversion_prior_False',
                  'MaskedMimic_Inversion_prior_True',
                  # 'MaskedMimic_Prior_Only_prior_True',
                  # 'MaskedMimic_Finetune_prior_False',
                  # 'MaskedMimic_Finetune_prior_True',
                  # 'PULSE_prior_False',
                  # 'AMP_prior_False',
                  # 'PureRL_prior_False',
                  # 'PPO_prior_False',
                  ]
env_name = 'direction_facing'
env_pretty_names = {
    'reach': 'Reach',
    'steering': 'Direction',
    env_name: 'Steering',
    'long_jump': 'Long Jump',
    'strike': 'Strike'
}

algo_pretty_names = {
    'MaskedMimic_Inversion_prior_False': 'Task Tokens (ours)',
    'MaskedMimic_Inversion_prior_True': 'Task Tokens (ours) + J.C.',
    'MaskedMimic_Prior_Only_prior_True': 'MaskedMimic (J.C. only)',
    'MaskedMimic_Finetune_prior_False': 'MaskedMimic Fine-Tune',
    'MaskedMimic_Finetune_prior_True': 'MaskedMimic Fine-Tune + J.C.',
    'PULSE_prior_False': 'PULSE',
    'AMP_prior_False': 'AMP',
    'PureRL_prior_False': 'PureRL',
    'PPO_prior_False': 'PPO',
}

latex_output_dir = f"{project_prefix}/latex_tables"
os.makedirs(latex_output_dir, exist_ok=True)


In [94]:
from analysis.analysis_utils import merge_rows
from glob import glob
from typing import List


def generate_intermediate_mean_std_df_ablations(file, keep_cols, renamed_keep_cols):
    df = pd.read_csv(file)
    df = df.dropna(axis=1, how='all')  # Drop empty columns
    df = df[keep_cols]
    df.columns = renamed_keep_cols
    df['mean'] = pd.to_numeric(df['mean'], errors='coerce')  # Convert to numeric, coerce errors to NaN
    df['std'] = pd.to_numeric(df['std'], errors='coerce')  # Convert to numeric, coerce errors to NaN
    df['value'] = df.apply(
        lambda row: f"{row['mean']:.2f} ± {row['std']:.2f}" if pd.notna(row['std']) else f"{row['mean']:.2f}",
        axis=1)
    df['algo_str'] = df['algo_type'] + '_prior_' + df['prior'].map(str) + '_current_pose_' + df['current_pose'].map(
        str) + '_bigger_' + df['bigger'].map(str)
    return df


def generate_combined_ablations_df(grouped_dir, project_prefix='FINALLY__'):
    # Read all CSV files
    csv_files = glob(os.path.join(grouped_dir, "*.csv"))
    keep_cols = ['algo_type', 'prior', 'bigger', 'current_pose',
                 'use_perturbations', 'reach_success_mean', 'reach_success_std']
    renamed_keep_cols = ['algo_type', 'prior', 'bigger', 'current_pose',
                         'use_perturbations', 'mean', 'std']
    # Dictionary to store data
    data = {}
    for file in csv_files:
        env_name = os.path.splitext(os.path.basename(file))[0].replace(project_prefix, '')  # Remove project prefix
        df = generate_intermediate_mean_std_df_ablations(file, keep_cols, renamed_keep_cols)
        df['env_perturb'] = env_name + '_perturb_' + df['use_perturbations'].map(str)

        for ep in df['env_perturb'].unique():
            data[ep] = \
                df[df['env_perturb'] == ep].set_index(['algo_str', 'algo_type', 'prior', 'bigger', 'current_pose', ])[
                    'value']
    # Combine data into a single DataFrame
    combined_df = pd.concat(data, axis=1).reset_index()
    return combined_df


def generate_ablations_table(final_df: pd.DataFrame, algos: List[str], envs: List[str], algo_rename_dict: dict,
                             env_rename_dict: dict):
    # different methods - PPO, AMP, PULSE, PRIOR_ONLY, INVERSION prior FALSE
    # 5 envs, no perturbations
    table_df = final_df.copy()
    table_df = table_df[table_df['algo_str'].map(lambda x: any([x.startswith(a) for a in algos]))]
    non_perturb_cols = [col for col in table_df.columns if 'perturb_False' in col]
    ablation_cols = ['prior', 'bigger', 'current_pose']
    table_df = table_df[['algo_str'] + non_perturb_cols + ablation_cols]

    renamed_cols = [col.replace('_perturb_False', '') for col in non_perturb_cols]
    # Rename columns
    table_df.columns = ['Method'] + renamed_cols + ablation_cols
    # Rename algo names
    table_df['Method'] = table_df['Method'].replace(algo_rename_dict)
    table_df = table_df[['Method'] + ablation_cols + envs]
    table_df = table_df.rename(columns=env_rename_dict)
    table_df['Method'] = table_df['Method'].map(lambda x: '_'.join(x.split('_')[:3]))
    table_df = table_df.fillna('-')
    del table_df['Method']
    return table_df


def post_process_ablations(df):
    df.rename(columns={'_perturb_False': 'direction_facing_perturb_False'}, inplace=True)
    df = merge_rows(df)
    return df

In [95]:
from analysis.analysis_utils import generate_grouped_csvs, generate_latex

GENERATE_RAW_CSV = False
GENERATE_GROUPED_CSV = True
if GENERATE_RAW_CSV:
    generate_raw_csvs(project_prefix, entity, filters)
if GENERATE_GROUPED_CSV:
    generate_grouped_csvs(project_prefix=project_prefix,
                          perturbation_types=(),
                          group_by_keys=["algo_type", "prior", "use_perturbations", "bigger", "current_pose"],
                          metrics_rename_dict=rename_metrics)

combined_df = generate_combined_ablations_df(f"{project_prefix}/grouped", project_prefix=project_prefix)

combined_ablations_df = generate_combined_ablations_df(f"{project_prefix}/grouped", project_prefix=project_prefix)

final_ablations_df = post_process_ablations(combined_ablations_df)

In [96]:
final_ablations_df

,algo_str,algo_type,prior,bigger,current_pose,direction_facing_perturb_False
0,MaskedMimic_Finetune_prior_False_current_pose_...,MaskedMimic_Finetune,False,False,False,93.54 ± 4.66
1,MaskedMimic_Finetune_prior_False_current_pose_...,MaskedMimic_Finetune,False,False,True,90.72 ± 5.42
2,MaskedMimic_Finetune_prior_False_current_pose_...,MaskedMimic_Finetune,False,True,False,92.68 ± 4.59
3,MaskedMimic_Finetune_prior_False_current_pose_...,MaskedMimic_Finetune,False,True,True,82.79 ± 11.85
4,MaskedMimic_Finetune_prior_True_current_pose_F...,MaskedMimic_Finetune,True,False,False,97.25 ± 1.38
5,MaskedMimic_Finetune_prior_True_current_pose_T...,MaskedMimic_Finetune,True,False,True,93.58 ± 0.80
6,MaskedMimic_Finetune_prior_True_current_pose_F...,MaskedMimic_Finetune,True,True,False,95.16 ± 4.41
7,MaskedMimic_Finetune_prior_True_current_pose_T...,MaskedMimic_Finetune,True,True,True,94.39 ± 3.85
8,MaskedMimic_Inversion_prior_False_current_pose...,MaskedMimic_Inversion,False,False,False,84.28 ± 7.72
9,MaskedMimic_Inversion_prior_False_current_pose...,MaskedMimic_Inversion,False,False,True,66.31 ± 13.11


In [121]:
ablations = generate_ablations_table(final_ablations_df, algos=relevant_algos, envs=[env_name],
                                     algo_rename_dict=algo_pretty_names,
                                     env_rename_dict=env_pretty_names)

ablations = ablations.iloc[ablations['Steering'].map(lambda x: float(x.split('±')[0])).argsort()[::-1]]
# rename J.C. & Bigger MLP & Using Current Pose & Steering Success Rate
ablations = ablations.rename(columns={'prior': 'Method', 'bigger': 'Bigger MLP', 'current_pose': 'Using Current Pose',
                          'Steering':'Steering Success Rate'})
ablations['Method'] = ablations['Method'].map(lambda x: 'Task Tokens (ours) + J.C.' if x else 'Task Tokens (ours)')

generate_latex(ablations, f"{latex_output_dir}/{env_name}_ablations.tex",
               caption="Ablation Study",
               label=f"tab:{env_name}_ablations")
ablations

,Method,Bigger MLP,Using Current Pose,Steering Success Rate
15,Task Tokens (ours) + J.C.,True,True,87.77 ± 7.14
14,Task Tokens (ours) + J.C.,True,False,87.58 ± 7.02
12,Task Tokens (ours) + J.C.,False,False,86.88 ± 6.65
8,Task Tokens (ours),False,False,84.28 ± 7.72
10,Task Tokens (ours),True,False,83.30 ± 10.06
13,Task Tokens (ours) + J.C.,False,True,79.47 ± 4.71
11,Task Tokens (ours),True,True,78.59 ± 8.93
9,Task Tokens (ours),False,True,66.31 ± 13.11


In [122]:
f"{latex_output_dir}/{env_name}_ablations.tex"

'LASTMIN_last_minute/latex_tables/direction_facing_ablations.tex'

In [119]:
ablations

,Method,Bigger MLP,Using Current Pose,Steering Success Rate
15,Task Tokens (ours) + J.C.,True,True,87.77 ± 7.14
14,Task Tokens (ours) + J.C.,True,False,87.58 ± 7.02
12,Task Tokens (ours) + J.C.,False,False,86.88 ± 6.65
8,Task Tokens (ours),False,False,84.28 ± 7.72
10,Task Tokens (ours),True,False,83.30 ± 10.06
13,Task Tokens (ours) + J.C.,False,True,79.47 ± 4.71
11,Task Tokens (ours),True,True,78.59 ± 8.93
9,Task Tokens (ours),False,True,66.31 ± 13.11
